# Module 4: Data Transformations

## Learning Objectives
By the end of this module, you will understand:
- Casting/converting data types
- Working with string functions
- Working with math functions
- Working with date/time functions
- Adding and modifying columns
- Creating complex transformations

---

## 1. Type Casting - Converting Data Types

Sometimes data comes in the wrong format and needs conversion.

In [ ]:
# Create sample data with wrong types
data = [
    ("1", "25.50", "2024-01-15"),
    ("2", "30.75", "2024-01-16"),
    ("3", "22.00", "2024-01-17")
]
df = spark.createDataFrame(data, ["id", "price", "date"])

print("Original DataFrame:")
df.show()
df.printSchema()

**Output:**
```
Original DataFrame:
+---+-----+----------+
| id|price|      date|
+---+-----+----------+
|  1| 25.5|2024-01-15|
|  2|30.75|2024-01-16|
|  3| 22.0|2024-01-17|
+---+-----+----------+

root
 |-- id: string (nullable = true)
 |-- price: string (nullable = true)
 |-- date: string (nullable = true)
```

Notice: All columns are strings! Let's convert them.

In [ ]:
# Method 1: Using cast()
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType, DateType

df_cast = df.select(
    col("id").cast(IntegerType()).alias("id"),
    col("price").cast(DoubleType()).alias("price"),
    col("date").cast(DateType()).alias("date")
)

print("After casting:")
df_cast.show()
df_cast.printSchema()

**Output:**
```
After casting:
+---+-----+----------+
| id|price|      date|
+---+-----+----------+
|  1| 25.5|2024-01-15|
|  2|30.75|2024-01-16|
|  3| 22.0|2024-01-17|
+---+-----+----------+

root
 |-- id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- date: date (nullable = true)
```

In [ ]:
# Method 2: Using string notation
df_cast2 = df.select(
    col("id").cast("integer"),
    col("price").cast("double"),
    col("date").cast("date")
)
df_cast2.printSchema()

### Common Type Conversions:
- `"string"` or `StringType()`
- `"integer"` or `IntegerType()`
- `"long"` or `LongType()`
- `"double"` or `DoubleType()`
- `"float"` or `FloatType()`
- `"boolean"` or `BooleanType()`
- `"date"` or `DateType()`
- `"timestamp"` or `TimestampType()`

---

## 2. String Functions

Common operations on text data.

In [ ]:
# Create sample data
data = [
    ("alice johnson", "  data science  "),
    ("bob smith", "software engineer"),
    ("charlie brown", "product manager")
]
df_names = spark.createDataFrame(data, ["full_name", "job_title"])
df_names.show(truncate=False)

**Output:**
```
+----------------+------------------+
|full_name       |job_title         |
+----------------+------------------+
|alice johnson   |  data science  |
|bob smith       |software engineer|
|charlie brown   |product manager   |
+----------------+------------------+
```

In [ ]:
from pyspark.sql.functions import upper, lower, trim, length, concat, substring, replace

# String function examples
df_strings = df_names.select(
    col("full_name"),
    upper(col("full_name")).alias("name_upper"),           # Convert to uppercase
    lower(col("full_name")).alias("name_lower"),           # Convert to lowercase
    length(col("full_name")).alias("name_length"),         # Get string length
    trim(col("job_title")).alias("job_title_trimmed"),    # Remove leading/trailing spaces
    substring(col("full_name"), 1, 5).alias("first_5_chars"), # Get first 5 characters
    replace(col("full_name"), " ", "_").alias("name_with_underscore") # Replace spaces
)

df_strings.show(truncate=False)

**Output:**
```
+---------------+-----------+-----------+-----------+------------------+---------------+--------------------+
|full_name      |name_upper |name_lower |name_length|job_title_trimmed|first_5_chars|name_with_underscore|
+---------------+-----------+-----------+-----------+------------------+---------------+--------------------+
|alice johnson  |ALICE JOHN |alice johnson|13        |data science    |alice        |alice_johnson      |
|bob smith      |BOB SMITH  |bob smith     |9         |software engineer|bob          |bob_smith          |
|charlie brown  |CHARLIE BR |charlie brown|13        |product manager  |charlie      |charlie_brown      |
+---------------+-----------+-----------+-----------+------------------+---------------+--------------------+
```

In [ ]:
# More string functions
from pyspark.sql.functions import split, concat_ws, initcap, ltrim, rtrim

df_more_strings = df_names.select(
    col("full_name"),
    initcap(col("full_name")).alias("title_case"),         # Title case
    split(col("full_name"), " ").alias("name_parts"),     # Split into array
    concat_ws("-", col("full_name")).alias("with_dashes"), # Join with separator
    ltrim(col("job_title")).alias("left_trimmed"),         # Trim left only
    rtrim(col("job_title")).alias("right_trimmed")         # Trim right only
)

df_more_strings.show(truncate=False)

**Output:**
```
+---------------+---------------+-------------------+---------------------+------------------+-------------------+
|full_name      |title_case     |name_parts         |with_dashes         |left_trimmed      |right_trimmed      |
+---------------+---------------+-------------------+---------------------+------------------+-------------------+
|alice johnson  |Alice Johnson  |[alice, johnson]   |alice johnson       |data science     |  data science    |
|bob smith      |Bob Smith      |[bob, smith]       |bob smith           |software engineer |software engineer |
|charlie brown  |Charlie Brown  |[charlie, brown]   |charlie brown       |product manager  |product manager   |
+---------------+---------------+-------------------+---------------------+------------------+-------------------+
```

---

## 3. Math Functions

Operations on numeric data.

In [ ]:
# Create sample data
data = [
    (1, 100, 25),
    (2, 150, 30),
    (3, 200, 45),
    (4, 75, 15)
]
df_numbers = spark.createDataFrame(data, ["id", "amount", "quantity"])
df_numbers.show()

**Output:**
```
+---+------+--------+
| id|amount|quantity|
+---+------+--------+
|  1|   100|      25|
|  2|   150|      30|
|  3|   200|      45|
|  4|    75|      15|
+---+------+--------+
```

In [ ]:
from pyspark.sql.functions import abs, round, ceil, floor, sqrt, pow, mod

# Math operations
df_math = df_numbers.select(
    col("id"),
    col("amount"),
    col("quantity"),
    (col("amount") * col("quantity")).alias("total"),        # Multiply
    (col("amount") / col("quantity")).alias("unit_price"),    # Divide
    round(col("amount") / col("quantity"), 2).alias("unit_price_rounded"), # Round
    ceil(col("amount") / col("quantity")).alias("unit_price_ceil"),       # Round up
    floor(col("amount") / col("quantity")).alias("unit_price_floor"),     # Round down
    sqrt(col("amount")).alias("sqrt_amount"),                 # Square root
    pow(col("quantity"), 2).alias("quantity_squared"),        # Power
    mod(col("amount"), col("quantity")).alias("remainder")   # Modulo
)

df_math.show()

**Output:**
```
+---+------+--------+------+----------+-----------+-----------+-----------+----------+------------------+----------+
| id|amount|quantity| total|unit_price|unit_price_rounded|unit_price_ceil|unit_price_floor|sqrt_amount|quantity_squared|remainder|
+---+------+--------+------+----------+-----------+-----------+-----------+----------+------------------+----------+
|  1|   100|      25| 2500|       4.0|        4.0|         4|          4|     10.0|                625|         0|
|  2|   150|      30| 4500|       5.0|        5.0|         5|          5|    12.25|                900|         0|
|  3|   200|      45| 9000|4.44444444|        4.44|         5|          4|    14.14|               2025|        20|
|  4|    75|      15| 1125|       5.0|        5.0|         5|          5|     8.66|                225|         0|
+---+------+--------+------+----------+-----------+-----------+-----------+----------+------------------+----------+
```

### Math Functions Summary:
- `abs()` - Absolute value
- `round(col, decimals)` - Round to N decimals
- `ceil()` - Round up
- `floor()` - Round down
- `sqrt()` - Square root
- `pow(col, exponent)` - Power
- `mod()` - Modulo (remainder)
- `sin()`, `cos()`, `tan()` - Trigonometric
- `log()`, `exp()` - Logarithm and exponential

---

## 4. Date and Time Functions

Working with temporal data.

In [ ]:
from pyspark.sql.functions import current_date, current_timestamp
from datetime import date, datetime

# Create sample date data
data = [
    (1, "2024-01-15", "2024-01-15 10:30:00"),
    (2, "2024-02-20", "2024-02-20 14:45:30"),
    (3, "2024-03-10", "2024-03-10 09:15:00")
]
df_dates = spark.createDataFrame(data, ["id", "date_str", "timestamp_str"])

# Cast to proper types
df_dates = df_dates.select(
    col("id"),
    col("date_str").cast("date").alias("date_col"),
    col("timestamp_str").cast("timestamp").alias("timestamp_col")
)

print("Original dates:")
df_dates.show()
df_dates.printSchema()

**Output:**
```
Original dates:
+---+----------+-------------------+
| id|  date_col|     timestamp_col|
+---+----------+-------------------+
|  1|2024-01-15|2024-01-15 10:30:00|
|  2|2024-02-20|2024-02-20 14:45:30|
|  3|2024-03-10|2024-03-10 09:15:00|
+---+----------+-------------------+

root
 |-- id: integer (nullable = true)
 |-- date_col: date (nullable = true)
 |-- timestamp_col: timestamp (nullable = true)
```

In [ ]:
from pyspark.sql.functions import year, month, dayofmonth, dayofweek, dayofyear, weekofyear
from pyspark.sql.functions import hour, minute, second, date_format

# Extract date components
df_date_parts = df_dates.select(
    col("date_col"),
    year(col("date_col")).alias("year"),
    month(col("date_col")).alias("month"),
    dayofmonth(col("date_col")).alias("day"),
    dayofweek(col("date_col")).alias("day_of_week"),      # 1=Sunday, 7=Saturday
    dayofyear(col("date_col")).alias("day_of_year"),
    weekofyear(col("date_col")).alias("week_of_year")
)

print("Date components:")
df_date_parts.show()

**Output:**
```
+----------+----+-----+---+-----------+-----------+-----------+
|  date_col|year|month|day|day_of_week|day_of_year|week_of_year|
+----------+----+-----+---+-----------+-----------+-----------+
|2024-01-15|2024|    1| 15|          2|         15|          3|
|2024-02-20|2024|    2| 20|          3|         51|          8|
|2024-03-10|2024|    3| 10|          1|         70|         11|
+----------+----+-----+---+-----------+-----------+-----------+
```

In [ ]:
# Extract time components
df_time_parts = df_dates.select(
    col("timestamp_col"),
    hour(col("timestamp_col")).alias("hour"),
    minute(col("timestamp_col")).alias("minute"),
    second(col("timestamp_col")).alias("second")
)

print("Time components:")
df_time_parts.show()

**Output:**
```
+-------------------+-----+------+------+
|     timestamp_col  |hour|minute|second|
+-------------------+-----+------+------+
|2024-01-15 10:30:00|  10|    30|     0|
|2024-02-20 14:45:30|  14|    45|    30|
|2024-03-10 09:15:00|   9|    15|     0|
+-------------------+-----+------+------+
```

In [ ]:
from pyspark.sql.functions import date_add, date_sub, datediff, add_months

# Date arithmetic
df_date_math = df_dates.select(
    col("date_col"),
    date_add(col("date_col"), 7).alias("date_plus_7_days"),
    date_sub(col("date_col"), 7).alias("date_minus_7_days"),
    datediff(current_date(), col("date_col")).alias("days_since"),
    add_months(col("date_col"), 3).alias("date_plus_3_months")
)

print("Date arithmetic:")
df_date_math.show(truncate=False)

**Output:**
```
+----------+------------------+------------------+----------+------------------+
|date_col  |date_plus_7_days  |date_minus_7_days |days_since|date_plus_3_months|
+----------+------------------+------------------+----------+------------------+
|2024-01-15|2024-01-22        |2024-01-08        |       37 |2024-04-15        |
|2024-02-20|2024-02-27        |2024-02-13        |        1 |2024-05-20        |
|2024-03-10|2024-03-17        |2024-03-03        |       12 |2024-06-10        |
+----------+------------------+------------------+----------+------------------+
```

In [ ]:
# Date formatting
from pyspark.sql.functions import date_format

df_formatted = df_dates.select(
    col("date_col"),
    date_format(col("date_col"), "yyyy-MM-dd").alias("format_iso"),
    date_format(col("date_col"), "MM/dd/yyyy").alias("format_us"),
    date_format(col("date_col"), "EEEE, MMMM d, yyyy").alias("format_long"),
    date_format(col("timestamp_col"), "yyyy-MM-dd HH:mm:ss").alias("format_timestamp")
)

print("Formatted dates:")
df_formatted.show(truncate=False)

**Output:**
```
+----------+----------+----------+-------------------------+------------------+
|date_col  |format_iso|format_us |format_long              |format_timestamp  |
+----------+----------+----------+-------------------------+------------------+
|2024-01-15|2024-01-15|01/15/2024|Monday, January 15, 2024 |2024-01-15 10:30:0|
|2024-02-20|2024-02-20|02/20/2024|Tuesday, February 20, 202|2024-02-20 14:45:3|
|2024-03-10|2024-03-10|03/10/2024|Sunday, March 10, 2024   |2024-03-10 09:15:0|
+----------+----------+----------+-------------------------+------------------+
```

---

## 5. Adding and Modifying Columns

Create new columns based on transformations.

In [ ]:
# Create sample DataFrame
data = [
    ("Alice", 70000),
    ("Bob", 85000),
    ("Charlie", 60000)
]
df_employees = spark.createDataFrame(data, ["name", "salary"])

print("Original:")
df_employees.show()

**Output:**
```
+-------+------+
|   name|salary|
+-------+------+
|  Alice| 70000|
|    Bob| 85000|
|Charlie| 60000|
+-------+------+
```

In [ ]:
# Add new columns
df_with_columns = df_employees.withColumn(
    "bonus",
    col("salary") * 0.1  # 10% bonus
).withColumn(
    "total_comp",
    col("salary") + col("bonus")
).withColumn(
    "salary_bracket",
    when(col("salary") >= 80000, "Senior")
    .when(col("salary") >= 60000, "Mid")
    .otherwise("Junior")
)

from pyspark.sql.functions import when
print("After adding columns:")
df_with_columns.show()

**Output:**
```
+-------+------+------+----------+---------------+
|   name|salary| bonus|total_comp|salary_bracket|
+-------+------+------+----------+---------------+
|  Alice| 70000| 7000|      77000|       Mid     |
|    Bob| 85000| 8500|      93500|       Senior  |
|Charlie| 60000| 6000|      66000|       Mid     |
+-------+------+------+----------+---------------+
```

---

## 6. Complex Transformation Pipeline

Chain multiple transformations together.

In [ ]:
# Create realistic sales data
data = [
    (1, "alice johnson", 1500.50, "2024-01-15"),
    (2, "bob smith", 2300.75, "2024-01-16"),
    (3, "charlie brown", 1800.00, "2024-01-17"),
    (4, "diana prince", 3200.25, "2024-01-18")
]
df_sales = spark.createDataFrame(data, ["order_id", "customer_name", "amount", "order_date"])

print("Original data:")
df_sales.show()

**Output:**
```
+--------+---------------+-------+----------+
|order_id|customer_name  |amount |order_date|
+--------+---------------+-------+----------+
|       1|alice johnson  |1500.5 |2024-01-15|
|       2|bob smith      |2300.75|2024-01-16|
|       3|charlie brown  |1800.0 |2024-01-17|
|       4|diana prince   |3200.25|2024-01-18|
+--------+---------------+-------+----------+
```

In [ ]:
# Complex transformation pipeline
from pyspark.sql.functions import when, col, upper, initcap, year, month, round as spark_round

df_transformed = df_sales \
    .select(
        col("order_id"),
        initcap(col("customer_name")).alias("customer_name"),  # Title case names
        col("amount").cast("double"),
        col("order_date").cast("date")
    ) \
    .withColumn(
        "tax",
        spark_round(col("amount") * 0.1, 2)  # Calculate tax (10%)
    ) \
    .withColumn(
        "total",
        col("amount") + col("tax")  # Add tax to total
    ) \
    .withColumn(
        "order_year",
        year(col("order_date"))
    ) \
    .withColumn(
        "order_month",
        month(col("order_date"))
    ) \
    .withColumn(
        "priority",
        when(col("total") >= 3000, "High")
        .when(col("total") >= 2000, "Medium")
        .otherwise("Low")
    )

print("After transformation:")
df_transformed.show(truncate=False)

**Output:**
```
+--------+---------------+------+----------+------+------+----------+-----------+--------+
|order_id|customer_name  |amount|order_date|  tax |total |order_year|order_month|priority|
+--------+---------------+------+----------+------+------+----------+-----------+--------+
|       1|Alice Johnson  |1500.5|2024-01-15|150.05|1650.55|    2024  |       1   |  Low   |
|       2|Bob Smith      |2300.75|2024-01-16|230.08|2530.83|    2024  |       1   | Medium |
|       3|Charlie Brown  |1800.0 |2024-01-17|180.0 |1980.0 |    2024  |       1   |  Low   |
|       4|Diana Prince   |3200.25|2024-01-18|320.03|3520.28|    2024  |       1   |  High  |
+--------+---------------+------+----------+------+------+----------+-----------+--------+
```

---

## 7. Mini Challenges

### Challenge 1: Type Conversions
1. Create a DataFrame with string columns
2. Cast id to integer
3. Cast price to double
4. Cast date to date type
5. Print schema to verify

### Challenge 2: String Transformations
1. Create DataFrame with person names
2. Convert to uppercase
3. Get first 3 characters
4. Get length of each name
5. Replace spaces with underscores

### Challenge 3: Math Operations
1. Create DataFrame with quantities and prices
2. Calculate total (quantity × price)
3. Calculate discount (10% of total)
4. Calculate final price (total - discount)
5. Round final price to 2 decimals

### Challenge 4: Date Transformations
1. Create DataFrame with dates
2. Extract year, month, day
3. Add 30 days to each date
4. Calculate days since today
5. Format as "MM/DD/YYYY"

---

## Key Takeaways

✅ **cast()** - Convert between data types

✅ **upper(), lower(), trim()** - Common string operations

✅ **round(), ceil(), floor()** - Math operations

✅ **year(), month(), day()** - Extract date parts

✅ **withColumn()** - Add/modify columns

✅ **when()...otherwise()** - Conditional logic

✅ **Chain transformations** - Build pipelines

---

## Next Steps
In Module 5, we'll learn **data cleaning** - filtering rows, handling nulls, and removing duplicates!